In [1]:
!pip install matplotlib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Dimensionless Constants ---
# Distance: 1 AU (Astronomical Unit)
# Time: 1 Earth Year
# Mass: 1 Solar Mass
G = 4 * np.pi**2 

def compute_accelerations(positions, masses):
    """
    Computes the gravitational acceleration for all bodies in 3D space.
    positions: numpy array of shape (N, 3)
    masses: numpy array of shape (N,)
    """
    N = len(masses)
    accelerations = np.zeros((N, 3))
    
    for i in range(N):
        for j in range(N):
            if i != j:
                r_vec = positions[j] - positions[i]
                r_mag = np.linalg.norm(r_vec)
                # Apply Newton's Law of Universal Gravitation
                accelerations[i] += G * masses[j] * r_vec / (r_mag**3)
                
    return accelerations

def rk4_step(positions, velocities, masses, dt):
    """
    Advances the system by one time step using 4th-Order Runge-Kutta.
    """
    k1_v = compute_accelerations(positions, masses)
    k1_r = velocities
    
    k2_v = compute_accelerations(positions + 0.5 * dt * k1_r, masses)
    k2_r = velocities + 0.5 * dt * k1_v
    
    k3_v = compute_accelerations(positions + 0.5 * dt * k2_r, masses)
    k3_r = velocities + 0.5 * dt * k2_v
    
    k4_v = compute_accelerations(positions + dt * k3_r, masses)
    k4_r = velocities + dt * k3_v
    
    new_positions = positions + (dt / 6.0) * (k1_r + 2*k2_r + 2*k3_r + k4_r)
    new_velocities = velocities + (dt / 6.0) * (k1_v + 2*k2_v + 2*k3_v + k4_v)
    
    return new_positions, new_velocities

# --- Initialization ---
# Masses: [Sun, Earth, Mars, Spaceship] (Spaceship mass is negligible)
masses = np.array([1.0, 3.003e-6, 3.213e-7, 0.0])

# Initial Positions (x, y, z) in AU
# Assuming a simplified starting alignment on the x-axis for Earth and Mars
positions = np.array([
    [0.0, 0.0, 0.0],       # Sun at origin
    [1.0, 0.0, 0.0],       # Earth at 1 AU
    [1.524, 0.0, 0.0],     # Mars at 1.524 AU
    [1.0, 0.0, 0.0]        # Spaceship starting at Earth
])

# Initial Velocities (vx, vy, vz) in AU/Year
# Circular orbit velocity is roughly v = sqrt(G * M_sun / r)
v_earth = np.sqrt(G * masses[0] / 1.0)
v_mars = np.sqrt(G * masses[0] / 1.524)

# Spaceship requires Earth's velocity + a departure delta-v for Hohmann transfer
# To be optimized later in the project
v_spacecraft = v_earth + 0.6  

velocities = np.array([
    [0.0, 0.0, 0.0],          # Sun (stationary in this frame)
    [0.0, v_earth, 0.0],      # Earth moving along y-axis
    [0.0, v_mars, 0.0],       # Mars moving along y-axis
    [0.0, v_spacecraft, 0.0]  # Spaceship departure velocity
])

# Simulation parameters
dt = 0.001  # Time step in years
total_time = 1.0  # Simulate for 1 Earth year
num_steps = int(total_time / dt)

# Arrays to store trajectory history for 3D visualization
history_pos = np.zeros((num_steps, len(masses), 3))

# Main simulation loop
for step in range(num_steps):
    history_pos[step] = positions
    positions, velocities = rk4_step(positions, velocities, masses, dt)


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


/tmp/ipykernel_12006/3296332854.py:27: RuntimeWarning: invalid value encountered in divide
  accelerations[i] += G * masses[j] * r_vec / (r_mag**3)
